1. Import Fuzzy dan Pengaturan Variabel Fuzzy

In [1]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

# 1. Definisi Variabel Input (Antecedents) dan Output (Consequent)
confidence = ctrl.Antecedent(np.arange(0, 1.01, 0.01), 'confidence')
area_ratio = ctrl.Antecedent(np.arange(0, 1.01, 0.01), 'area_ratio')
capacity = ctrl.Consequent(np.arange(0, 101, 1), 'capacity')

# 2. Fungsi Keanggotaan (Membership Functions)
confidence['low'] = fuzz.trimf(confidence.universe, [0, 0, 0.5])
confidence['medium'] = fuzz.trimf(confidence.universe, [0.3, 0.6, 0.8])
confidence['high'] = fuzz.trimf(confidence.universe, [0.6, 1.0, 1.0])

area_ratio['small'] = fuzz.trimf(area_ratio.universe, [0, 0, 0.35])
area_ratio['medium'] = fuzz.trimf(area_ratio.universe, [0.25, 0.5, 0.75])
area_ratio['large'] = fuzz.trimf(area_ratio.universe, [0.6, 1.0, 1.0])

capacity['aman'] = fuzz.trimf(capacity.universe, [0, 0, 45])
capacity['hampir_penuh'] = fuzz.trimf(capacity.universe, [35, 60, 80])
capacity['penuh'] = fuzz.trimf(capacity.universe, [70, 100, 100])

print("[INFO] Variabel & Membership Functions Fuzzy Siap.")

[INFO] Variabel & Membership Functions Fuzzy Siap.


2. Aturan Dan Komuptasi fuzzy

In [2]:
# 3. Rules Fuzzy Mamdani
rule1 = ctrl.Rule(area_ratio['small'], capacity['aman'])
rule2 = ctrl.Rule(area_ratio['medium'] & confidence['high'], capacity['hampir_penuh'])
rule3 = ctrl.Rule(area_ratio['medium'] & ~confidence['high'], capacity['aman'])
rule4 = ctrl.Rule(area_ratio['large'] & (confidence['medium'] | confidence['high']), capacity['penuh'])

capacity_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4])

# 4. Fungsi Komputasi
def compute_fuzzy_capacity(conf_val, area_val):
    sim = ctrl.ControlSystemSimulation(capacity_ctrl)
    sim.input['confidence'] = float(conf_val)
    sim.input['area_ratio'] = float(area_val)

    try:
        sim.compute()
        score = sim.output['capacity']
    except Exception:
        score = (conf_val * 0.3 + area_val * 0.7) * 100

    if score >= 75:
        status = "PENUH"
        buzzer = True
    elif score >= 45:
        status = "HAMPIR PENUH"
        buzzer = False
    else:
        status = "AMAN"
        buzzer = False

    return {
        "capacity_percentage": round(score, 2),
        "status": status,
        "trigger_buzzer": buzzer
    }

print("[INFO] Engine Fuzzy Mamdani Berhasil Didefinisikan.")

[INFO] Engine Fuzzy Mamdani Berhasil Didefinisikan.


3. Pengujian Fuzzy


In [3]:
# Uji Coba Komputasi Fuzzy
test_result = compute_fuzzy_capacity(conf_val=0.85, area_val=0.65)
print("=== HASIL UJI MAMDANI ENGINE ===")
print("Confidence Input : 0.85 (85%)")
print("Area Ratio Input : 0.65 (65%)")
print("Kapasitas Hasil  :", test_result["capacity_percentage"], "%")
print("Status Container :", test_result["status"])
print("Buzzer Trigger   :", test_result["trigger_buzzer"])

=== HASIL UJI MAMDANI ENGINE ===
Confidence Input : 0.85 (85%)
Area Ratio Input : 0.65 (65%)
Kapasitas Hasil  : 43.22 %
Status Container : AMAN
Buzzer Trigger   : False
